Prueba

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [2]:
import pandas as pd

datos = pd.read_csv("C:\\Users\\wamt1\\OneDrive\\Escritorio\\computadorNuevo2\\datosNarmax\\24pasos_mlp_consumption.csv")

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [ ]:
datos.head()

,temp,zone1,zone2,zone3,hour,e
date,,,,,,
2017-01-01 00:00:00,-2.051356,0.151445,-0.851669,0.033500,-1.660858,NaN
2017-01-01 00:10:00,-2.074813,-0.433079,-0.211097,0.016502,-1.660858,NaN
2017-01-01 00:20:00,-2.091152,-0.527709,-0.283791,-0.055068,-1.660858,NaN
2017-01-01 00:30:00,-2.122213,-0.651648,-0.411186,-0.174054,-1.660858,NaN
2017-01-01 00:40:00,-2.154568,-0.774750,-0.507632,-0.244729,-1.660858,NaN


In [4]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Espacio de búsqueda

In [5]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas GRU
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades GRU
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes GRU, es decir [observaciones, retardos, caracteristicas]

In [6]:
futuros = 24
pasados  = 12

In [7]:
datosX = []
datosY = []
for i in range(pasados, len(datos) - futuros + 1):
  datosX.append(datos.iloc[i-pasados:i, 0:datos.shape[1]])
  datosY.append(datos.iloc[i+futuros-1:i+futuros, 1])


In [8]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (52381, 12, 6)
Dimensiones de Y: (52381, 1)


In [9]:
inputs = datosX.shape[1] * datosX.shape[2]
datosX = datosX.reshape(datosX.shape[0], inputs)

In [10]:
print("Dimensiones de X después de rehape:", datosX.shape)

Dimensiones de X después de rehape: (52381, 72)


Se dividen nuevamente los conjuntos de datos

In [11]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (36666, 72)
Las dimensiones de testX son:  (10529, 72)
Las dimensiones de valX son:  (5186, 72)


In [12]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (36666, 1)
Las dimensiones de testY son:  (10529, 1)
Las dimensiones de valY son:  (5186, 1)


Se crean métricas para medir desempeño

In [13]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

Versión Final


In [14]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(testX.shape[1],)))
    if (params['layers'] == 1):
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(Dense(units=params['units'], activation=params['activation']))
          model.add(Dropout(params['dropout']))
      model.add(Dense(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(testX, testY, epochs=128,
                        validation_split=0.3,
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])

    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [15]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=12, trials=trials, rstate=np.random.default_rng(42))

print(best)

  0%|          | 0/12 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

58/58 - 4s - 72ms/step - ia: 0.2821 - loss: 1.2364 - mae: 0.9300 - rmse: 1.1099 - smape: 1.4416 - val_ia: 0.3447 - val_loss: 1.6637 - val_mae: 1.1007 - val_rmse: 1.2758 - val_smape: 1.6677

Epoch 2/128                                           

58/58 - 0s - 4ms/step - ia: 0.2649 - loss: 1.1097 - mae: 0.8774 - rmse: 1.0522 - smape: 1.4706 - val_ia: 0.3444 - val_loss: 1.4304 - val_mae: 1.0187 - val_rmse: 1.1823 - val_smape: 1.7004

Epoch 3/128                                           

58/58 - 0s - 4ms/step - ia: 0.2600 - loss: 1.0323 - mae: 0.8388 - rmse: 1.0146 - smape: 1.4719 - val_ia: 0.3418 - val_loss: 1.2794 - val_mae: 0.9618 - val_rmse: 1.1178 - val_smape: 1.7371

Epoch 4/128                                           

58/58 - 0s - 4ms/step - ia: 0.2524 - loss: 1.0034 - mae: 0.8225 - rmse: 1.0007 - smape: 1.4824 - val_ia: 0.3384 - val_loss: 1.1789 - val_mae: 0.9216 - val_rmse: 1.0730 - val_smape: 1.7748

Epoch 5/128        

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                      

461/461 - 5s - 11ms/step - ia: 0.5865 - loss: 0.4269 - mae: 0.5069 - rmse: 0.6380 - smape: 0.9431 - val_ia: 0.2471 - val_loss: 0.5169 - val_mae: 0.5523 - val_rmse: 0.6097 - val_smape: 0.8979

Epoch 2/128                                                                      

461/461 - 1s - 3ms/step - ia: 0.6678 - loss: 0.3093 - mae: 0.4271 - rmse: 0.5466 - smape: 0.7860 - val_ia: 0.2374 - val_loss: 0.7231 - val_mae: 0.6435 - val_rmse: 0.6987 - val_smape: 0.8868

Epoch 3/128                                                                      

461/461 - 1s - 3ms/step - ia: 0.6711 - loss: 0.3041 - mae: 0.4240 - rmse: 0.5403 - smape: 0.7790 - val_ia: 0.2122 - val_loss: 0.8248 - val_mae: 0.7046 - val_rmse: 0.7466 - val_smape: 0.9397

Epoch 4/128                                                                      

461/461 - 1s - 3ms/step - ia: 0.6705 - loss: 0.3026 - mae: 0.4243 - rmse: 0.5393 - smape: 0.78

...

461/461 - 9s - 20ms/step - ia: 0.3019 - loss: 1.0302 - mae: 0.8282 - rmse: 1.0012 - smape: 1.4287 - val_ia: 0.1967 - val_loss: 0.7095 - val_mae: 0.7010 - val_rmse: 0.7320 - val_smape: 1.3626

Epoch 2/128                                                                     

461/461 - 1s - 2ms/step - ia: 0.3715 - loss: 0.7966 - mae: 0.7196 - rmse: 0.8813 - smape: 1.3183 - val_ia: 0.2078 - val_loss: 0.5115 - val_mae: 0.5992 - val_rmse: 0.6290 - val_smape: 1.0871

Epoch 3/128                                                                     

461/461 - 1s - 3ms/step - ia: 0.4411 - loss: 0.6625 - mae: 0.6508 - rmse: 0.8025 - smape: 1.2042 - val_ia: 0.2273 - val_loss: 0.4069 - val_mae: 0.5245 - val_rmse: 0.5584 - val_smape: 0.9251

Epoch 4/128                                                                     

461/461 - 1s - 2ms/step - ia: 0.5100 - loss: 0.5458 - mae: 0.5891 - rmse: 0.7269 - smape: 1.0932 - val_ia: 0.2289 - val_loss: 0.3832 - val_mae: 0.5016 - val_rmse: 0.5378 - val_smap

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                                                       

922/922 - 4s - 4ms/step - ia: 0.2725 - loss: 2.6636 - mae: 1.2470 - rmse: 1.5586 - smape: 1.5179 - val_ia: 0.1096 - val_loss: 1.6306 - val_mae: 1.0791 - val_rmse: 1.0951 - val_smape: 1.6114

Epoch 2/128                                                                       

922/922 - 2s - 2ms/step - ia: 0.2770 - loss: 2.4684 - mae: 1.2041 - rmse: 1.4987 - smape: 1.5122 - val_ia: 0.1103 - val_loss: 1.6026 - val_mae: 1.0695 - val_rmse: 1.0854 - val_smape: 1.6099

Epoch 3/128                                                                       

922/922 - 2s - 2ms/step - ia: 0.2776 - loss: 2.4356 - mae: 1.1982 - rmse: 1.4985 - smape: 1.5083 - val_ia: 0.1112 - val_loss: 1.5735 - val_mae: 1.0597 - val_rmse: 1.0754 - val_smape: 1.6091

Epoch 4/128                                                                       

922/922 - 2s - 2ms/step - ia: 0.2794 - loss: 2.3598 - mae: 1.1810 - rmse: 1.4694 - smape: 1

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



58/58 - 3s - 46ms/step - ia: 0.2372 - loss: 1.0808 - mae: 0.8571 - rmse: 1.0328 - smape: 1.4989 - val_ia: 0.3733 - val_loss: 0.7787 - val_mae: 0.7386 - val_rmse: 0.8715 - val_smape: 1.4332

Epoch 2/128                                                                        

58/58 - 0s - 5ms/step - ia: 0.3816 - loss: 0.7113 - mae: 0.6880 - rmse: 0.8410 - smape: 1.3033 - val_ia: 0.4670 - val_loss: 0.4902 - val_mae: 0.5925 - val_rmse: 0.6896 - val_smape: 1.1014

Epoch 3/128                                                                        

58/58 - 0s - 3ms/step - ia: 0.4889 - loss: 0.5726 - mae: 0.6062 - rmse: 0.7545 - smape: 1.1453 - val_ia: 0.5348 - val_loss: 0.4041 - val_mae: 0.5302 - val_rmse: 0.6242 - val_smape: 0.9757

Epoch 4/128                                                                        

58/58 - 0s - 4ms/step - ia: 0.5519 - loss: 0.4861 - mae: 0.5548 - rmse: 0.6966 - smape: 1.0334 - val_ia: 0.5727 - val_loss: 0.3574 - val_mae: 0.4884 - val_rmse: 0.5871 - val_sma

In [16]:
print(best)

{'activation': 1, 'batch': 1, 'dropout': 0.4, 'layers': 2.0, 'learning_rate': 8.360431152477965e-05, 'units': 4}


In [17]:
#{'activation': 1, 'batch': 1, 'dropout': 0.4, 'layers': 2.0, 'learning_rate': 8.360431152477965e-05, 'units': 4}